# Barekatrezaei et al. (ESWA 2025): Laplace-Beltrami spectra + dual-path CNN

Our own reproduction of the first paper, from `reproduce_eswa2025.py` (written from the paper text).
Protocol: 5-fold CV on Train+Validation (130), stacking meta-learner, report on the official Test split (70).
Positive class = malignant. The authors' original code will be added at the end for comparison.

## Paper results (Table 3)

In [ ]:
import pandas as pd
paper = pd.DataFrame({
    "AUC": [0.8233, 0.7771, 0.7867, 0.8725, 0.9350],
    "Acc": [0.7429, 0.6571, 0.6857, 0.7286, 0.8429],
    "Prec": [0.7037, 0.6667, 0.6552, 0.6842, 0.7959],
    "Sens": [0.9500, 0.8000, 0.9500, 0.9750, 0.9750],
    "Spec": [0.4667, 0.4667, 0.3333, 0.4000, 0.6667],
    "F1": [0.8085, 0.7273, 0.7755, 0.8041, 0.8764],
}, index=["SVM", "ExtraTrees", "CNN", "Voting", "Stacking"])
paper

## Features: radiomic, Laplace-Beltrami eigenvalues, 128x64x80 patches (cached)

In [ ]:
import sys; sys.path.insert(0, "..")
import reproduce_eswa2025 as r
# r.build_cache() takes about 1.7 h; the Kaggle runs below already did it (see ../kaggle/README.md)

## Our results (Kaggle GPU runs, stored in `../results/eswa2025/`)

Seed 0 comes from the first run (gradient-boosting stacker only); seeds 1 and 2 also have a logistic-regression stacker. Re-run with `r.main(epochs=60, seed=s)` (about 1 h on a GPU).

In [ ]:
from pathlib import Path
R = Path("../results/eswa2025")
runs = {f"seed {s}": pd.read_csv(R / f, index_col=0).rename(index={"Stacking": "Stacking-HGB"}) for s, f in [(0, "table3_seed0.csv"), (1, "results_seed1.csv"), (2, "results_seed2.csv")]}
ours = runs["seed 0"]
pd.DataFrame({k: v["AUC"] for k, v in runs.items()}).round(4)  # AUC per seed

## Ours minus paper

In [ ]:
(ours.rename(index={"Stacking-HGB": "Stacking"}) - paper).round(4)

## Ablations (paper Tables 4-8), seed 0\n\nRows are `table config [aggregator]`, aggregator = Voting, Stacking-HGB or Stacking-LR. The paper's Isomap/2-dim choice and the importance of the Laplace-Beltrami features do **not** reproduce.

In [ ]:
abl = pd.read_csv(R / "ablations.csv", index_col=0)
abl

## What we found about the Laplace-Beltrami features

5-fold CV on the 130 Train+Validation volumes, logistic regression, mean of 5 fold splits:

| eigenvalue version | corr. with lesion volume | LB only | + radiomic |
|---|---|---|---|
| raw log (ours) | -0.98 | 0.725 | 0.823 |
| area-normalised log(lambda*A) | -0.31 | 0.671 | 0.755 |
| ratio log(lambda_k/lambda_1) | 0.16 | 0.501 | 0.774 |
| radiomic only | | | 0.761 |

Raw eigenvalues help mostly as a lesion-size proxy; normalising makes them worse. The cause of the gap to the paper (stacking AUC 0.935 vs our 0.68-0.82) is still unknown.

## Authors' original code\n\nTODO: paste/import the code sent by the first author and compare against the tables above (start with the Laplace-Beltrami computation).